# Meta-learner (S/T/X-learner) 人工データ実験 — 状況1〜6

卒論7章(数値実験)の再現。`metalearner/` パッケージの関数を、状況ごとに同じ手順で呼ぶだけのノートブック。
状況ごとのパラメータ(次元・処置割合・応答関数)は `metalearner/data_synthetic.py` の `SITUATIONS` にまとめてある。

計算量の注意: 卒論と同じ設定(N=100,000、ハイパラ探索n_iter=50(XGB)/30(RF))はかなり時間がかかる。
まず動作を確認したいときは、下のセルの `N`, `N_ITER_XGB`, `N_ITER_RF` を小さくして試すとよい。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from metalearner.data_synthetic import simulate_situation, make_df, SituationConfig, SITUATIONS
from metalearner.learners import (
    search_xgb, search_rf,
    fit_s_learner, predict_s_learner,
    fit_t_learner, predict_t_learner,
    fit_x_learner, predict_x_learner,
)
from metalearner.evaluate import RMSE, MAE
from metalearner.shap_utils import compute_ite_shap, plot_ite_shap

In [ ]:
# ここを書き換えれば、試す状況や計算量を調整できる
SITUATIONS_TO_RUN = [1, 2, 3, 4, 5, 6]
N = None            # Noneなら卒論と同じ設定(各状況のSITUATIONSの値)を使う。小さくして試す場合は例えば 5000
N_ITER_XGB = 50      # XGBoostのランダムサーチ試行回数(卒論は50)
N_ITER_RF = 30       # ランダムフォレストのランダムサーチ試行回数(卒論は30)
CV = 5               # 交差検証の分割数(卒論は5)

In [ ]:
def run_situation(situation: int):
    """1つの状況について、S/T/X-learner(XGBoost・RFそれぞれ)を学習しRMSE/MAEを計算する"""
    cfg = SITUATIONS[situation]
    if N is not None:
        cfg = SituationConfig(d=cfg.d, e_const=cfg.e_const, structure=cfg.structure, N=N)

    X, tau_true, Y0, Y1, W, Y = simulate_situation(situation, config=cfg)
    df = make_df(X, W, Y, tau_true, Y0, Y1)

    df_train, df_test = train_test_split(df, test_size=0.5, random_state=0, stratify=df["W"])

    feature_cols = [c for c in df.columns if c.startswith("X")]
    X_train = df_train[feature_cols]
    W_train = df_train["W"].to_numpy()
    Y_train = df_train["Y"].to_numpy()
    X_test = df_test[feature_cols]
    tau_test = df_test["tau"].to_numpy()

    # ---- S-learner ----
    s_model_xgb, s_model_rf = fit_s_learner(X_train, W_train, Y_train, n_iter=N_ITER_XGB, cv=CV)
    s_tau_xgb, _, _ = predict_s_learner(s_model_xgb, X_test)
    s_tau_rf, _, _ = predict_s_learner(s_model_rf, X_test)

    # ---- T-learner ----
    t_model_0_xgb, t_model_1_xgb, t_model_0_rf, t_model_1_rf = fit_t_learner(
        X_train, W_train, Y_train, n_iter=N_ITER_XGB, cv=CV
    )
    t_tau_xgb, _, _ = predict_t_learner(t_model_0_xgb, t_model_1_xgb, X_test)
    t_tau_rf, _, _ = predict_t_learner(t_model_0_rf, t_model_1_rf, X_test)

    # ---- X-learner (T-learnerのモデルを使って学習する) ----
    x_tau0_xgb, x_tau1_xgb, x_prop_xgb = fit_x_learner(
        X_train, W_train, Y_train, t_model_0_xgb, t_model_1_xgb,
        search_fn=search_xgb, n_iter=N_ITER_XGB, cv=CV,
    )
    x_tau_xgb, _, _, _ = predict_x_learner(x_tau0_xgb, x_tau1_xgb, x_prop_xgb, X_test)

    x_tau0_rf, x_tau1_rf, x_prop_rf = fit_x_learner(
        X_train, W_train, Y_train, t_model_0_rf, t_model_1_rf,
        search_fn=search_rf, n_iter=N_ITER_RF, cv=CV,
    )
    x_tau_rf, _, _, _ = predict_x_learner(x_tau0_rf, x_tau1_rf, x_prop_rf, X_test)

    # ---- 結果をまとめる ----
    rows = [
        {"situation": situation, "learner": "S", "model": "xgb", "RMSE": RMSE(tau_test, s_tau_xgb), "MAE": MAE(tau_test, s_tau_xgb)},
        {"situation": situation, "learner": "S", "model": "rf",  "RMSE": RMSE(tau_test, s_tau_rf),  "MAE": MAE(tau_test, s_tau_rf)},
        {"situation": situation, "learner": "T", "model": "xgb", "RMSE": RMSE(tau_test, t_tau_xgb), "MAE": MAE(tau_test, t_tau_xgb)},
        {"situation": situation, "learner": "T", "model": "rf",  "RMSE": RMSE(tau_test, t_tau_rf),  "MAE": MAE(tau_test, t_tau_rf)},
        {"situation": situation, "learner": "X", "model": "xgb", "RMSE": RMSE(tau_test, x_tau_xgb), "MAE": MAE(tau_test, x_tau_xgb)},
        {"situation": situation, "learner": "X", "model": "rf",  "RMSE": RMSE(tau_test, x_tau_rf),  "MAE": MAE(tau_test, x_tau_rf)},
    ]

    # SHAP用にS-learner(RF)のモデルとテストデータも返しておく
    return pd.DataFrame(rows), s_model_rf, X_test

In [ ]:
all_results = []
s_model_rf_by_situation = {}
x_test_by_situation = {}

for situation in SITUATIONS_TO_RUN:
    print(f"=== 状況{situation} (d={SITUATIONS[situation].d}, e(x)={SITUATIONS[situation].e_const}) ===")
    results_df, s_model_rf, X_test = run_situation(situation)
    all_results.append(results_df)
    s_model_rf_by_situation[situation] = s_model_rf
    x_test_by_situation[situation] = X_test
    print(results_df)
    print()

summary = pd.concat(all_results, ignore_index=True)
summary

## SHAP値(ITEへの寄与度) — 状況1の例

S-learner(ランダムフォレスト)を対象に、W=1での予測とW=0での予測のSHAP値の差分をITEのSHAP値とみなす。
別の状況を見たい場合は `example_situation` を書き換える。

In [ ]:
example_situation = 1
shap_values_ite, X_sampled = compute_ite_shap(
    s_model_rf_by_situation[example_situation],
    x_test_by_situation[example_situation],
)
plot_ite_shap(shap_values_ite, X_sampled)